# BigEarthNet-S1 Dataset Exploration

This notebook explores the BigEarthNet-S1 (Sentinel-1 SAR) dataset. It scans the directory structure, counts the total number of patches, checks image properties (dimensions, bands, data types), verifies the presence of VV and VH bands, checks the pixel value ranges, and plots random SAR patches using `matplotlib`.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import configuration settings
import sys
sys.path.append(os.path.abspath('..'))
from config.settings import DATA_DIR, VV_BAND_SUFFIX, VH_BAND_SUFFIX, setup_logging

logger = setup_logging("DatasetExploration")
logger.info(f"Using dataset directory: {DATA_DIR}")

## 1. Directory Structure Analysis & Patch Counting

We will scan the dataset folder to understand its directory structure and count the total number of patches. BigEarthNet-S1 is typically organized in a nested structure where patches are grouped into subfolders.

In [ ]:
# Scan top-level directories
top_dirs = [d for d in DATA_DIR.iterdir() if d.is_dir()]
print(f"Root dataset directory: {DATA_DIR}")
print(f"Number of group directories: {len(top_dirs)}")

# Count total patch folders and collect their paths
patch_paths = []
for group_dir in top_dirs:
    for patch_dir in group_dir.iterdir():
        if patch_dir.is_dir():
            patch_paths.append(patch_dir)

total_patches = len(patch_paths)
print(f"Total SAR image patches detected: {total_patches}")

## 2. Band File and Extension Identification

Let's identify the file types (extensions) and the polarization bands (VV and VH) for a sample patch.

In [ ]:
if total_patches > 0:
    sample_patch = patch_paths[0]
    print(f"Sample patch folder: {sample_patch.name}")
    
    # List files in the sample patch folder
    sample_files = list(sample_patch.iterdir())
    for f in sample_files:
        print(f"  - File: {f.name} (Extension: {f.suffix})")
        
    # Detect VV and VH bands
    vv_file = next((f for f in sample_files if f.name.endswith(VV_BAND_SUFFIX)), None)
    vh_file = next((f for f in sample_files if f.name.endswith(VH_BAND_SUFFIX)), None)
    
    print(f"\nDetected VV band file: {vv_file.name if vv_file else 'None'}")
    print(f"Detected VH band file: {vh_file.name if vh_file else 'None'}")
else:
    print("No patches found to analyze.")

## 3. Image Dimensions and Pixel Value Ranges

We will load the VV and VH band GeoTIFF files for a sample patch using `rasterio` (falling back to `PIL` if needed) to print their dimensions and check their pixel values (min, max, mean, std).

In [ ]:
def load_band_data(file_path):
    """Loads band image data as a numpy array."""
    try:
        import rasterio
        with rasterio.open(file_path) as src:
            data = src.read(1)
            meta = {"width": src.width, "height": src.height, "dtype": src.dtypes[0]}
            return data, meta
    except ImportError:
        from PIL import Image
        img = Image.open(file_path)
        data = np.array(img)
        meta = {"width": img.width, "height": img.height, "dtype": str(data.dtype)}
        return data, meta

if total_patches > 0 and vv_file and vh_file:
    vv_data, vv_meta = load_band_data(vv_file)
    vh_data, vh_meta = load_band_data(vh_file)
    
    print("=== VV Band Properties ===")
    print(f"Dimensions: {vv_meta['width']}x{vv_meta['height']}")
    print(f"Data Type: {vv_meta['dtype']}")
    print(f"Raw Pixel Value Range: Min={vv_data.min():.4f}, Max={vv_data.max():.4f}")
    print(f"Raw Statistics: Mean={vv_data.mean():.4f}, Std={vv_data.std():.4f}")
    
    print("\n=== VH Band Properties ===")
    print(f"Dimensions: {vh_meta['width']}x{vh_meta['height']}")
    print(f"Data Type: {vh_meta['dtype']}")
    print(f"Raw Pixel Value Range: Min={vh_data.min():.4f}, Max={vh_data.max():.4f}")
    print(f"Raw Statistics: Mean={vh_data.mean():.4f}, Std={vh_data.std():.4f}")
else:
    print("Sample files not available.")

## 4. Visualizing SAR Images

Sentinel-1 SAR backscatter coefficients are usually in decibel (dB) or linear scale. They appear black-and-white. We can plot the VV and VH bands side-by-side. To improve visibility, we apply standard percentile clipping to remove extreme outliers.

In [ ]:
def plot_sar_patch(vv_img, vh_img, patch_name):
    """Plots VV and VH channels side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # Standardize visualization by clipping 1% and 99% percentiles
    def clip_img(img):
        p1, p99 = np.percentile(img, [1, 99])
        return np.clip(img, p1, p99)

    axes[0].imshow(clip_img(vv_img), cmap='gray')
    axes[0].set_title(f"{patch_name} - VV Band")
    axes[0].axis('off')

    axes[1].imshow(clip_img(vh_img), cmap='gray')
    axes[1].set_title(f"{patch_name} - VH Band")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

if total_patches > 0 and vv_file and vh_file:
    plot_sar_patch(vv_data, vh_data, sample_patch.name)
else:
    print("Sample images not loaded.")

## 5. Visualizing Random SAR Patches

Let's select three random patches from the dataset and plot their VV and VH bands to understand the variance in SAR imagery across different geographic scenes.

In [ ]:
if total_patches >= 3:
    random_patches = random.sample(patch_paths, 3)
    for patch in random_patches:
        # Load band files
        vv_path = next(patch.glob(f"*{VV_BAND_SUFFIX}"), None)
        vh_path = next(patch.glob(f"*{VH_BAND_SUFFIX}"), None)
        
        if vv_path and vh_path:
            vvd, _ = load_band_data(vv_path)
            vhd, _ = load_band_data(vh_path)
            plot_sar_patch(vvd, vhd, patch.name)
        else:
            print(f"Missing bands in patch: {patch.name}")
else:
    print("Not enough patches available for random plotting.")

## 6. Global Dataset Statistics Analysis (Sampled)

To help write the preprocessing pipeline, we can analyze the distribution of pixel values across a subset of 100 random patches. This gives us a good estimate of the mean, standard deviation, and clipping bounds for VV and VH bands.

In [ ]:
num_samples = min(100, total_patches)
if num_samples > 0:
    sampled_patches = random.sample(patch_paths, num_samples)
    
    vv_means = []
    vv_stds = []
    vh_means = []
    vh_stds = []
    
    for patch in sampled_patches:
        vv_path = next(patch.glob(f"*{VV_BAND_SUFFIX}"), None)
        vh_path = next(patch.glob(f"*{VH_BAND_SUFFIX}"), None)
        
        if vv_path and vh_path:
            vvd, _ = load_band_data(vv_path)
            vhd, _ = load_band_data(vh_path)
            
            vv_means.append(vvd.mean())
            vv_stds.append(vvd.std())
            vh_means.append(vhd.mean())
            vh_stds.append(vhd.std())
            
    print(f"=== Sampled Statistics over {num_samples} patches ===")
    print(f"VV Mean estimate: {np.mean(vv_means):.4f}")
    print(f"VV Std estimate: {np.mean(vv_stds):.4f}")
    print(f"VH Mean estimate: {np.mean(vh_means):.4f}")
    print(f"VH Std estimate: {np.mean(vh_stds):.4f}")
else:
    print("No patches to generate statistics.")